# Simple Supervised Fine-Tuning: Hedge Fund Q&A

## 🎯 Problem Definition

**Business Problem**: Hedge fund analysts spend time answering repetitive questions about portfolio analysis, risk assessment, and investment strategies. We want to automate responses to common queries.

**ML Problem**: Given a hedge fund question (input), predict the expert analyst answer (output).

**Approach**: Supervised fine-tuning on labeled question-answer pairs.

## 📊 Dataset

- **Type**: Labeled data (CSV)
- **Size**: 20 question-answer pairs
- **Format**: Each row has a question (input) and answer (label)

## 🔄 Workflow

1. Load labeled training data
2. Load pre-trained **Databricks GPT-OSS-120B** model
3. Fine-tune using supervised learning
4. Evaluate on test questions
5. Save to Unity Catalog

**Model**: `databricks-gpt-oss-120b`  
**Catalog/Schema**: `users.yasamin_tari`

## Step 1: Install Dependencies

In [ ]:
# Install required libraries
%pip install transformers datasets torch pandas mlflow --quiet
dbutils.library.restartPython()

## Step 2: Load and Inspect Training Data

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset
import mlflow
from datetime import datetime

# Configuration
catalog = "users"
schema = "yasamin_tari"
model_name = "databricks-gpt-oss-120b"

print(f"Using catalog.schema: {catalog}.{schema}")
print(f"Model: {model_name}")

In [ ]:
# Upload training_data.csv to DBFS first, or load from local path
# For this demo, we'll create it inline

# Load the CSV data
df = pd.read_csv("/dbfs/FileStore/training_data.csv")

print(f"✓ Loaded {len(df)} training examples\n")
print("=== Sample Data ===")
print(f"\nQuestion: {df.iloc[0]['question']}")
print(f"\nAnswer: {df.iloc[0]['answer'][:200]}...")

display(df.head(3))

## Step 3: Prepare Data for Training

We'll format each example as: `Question: {question}\n\nAnswer: {answer}`

In [ ]:
# Format data: combine question and answer into training text
def format_example(row):
    return f"Question: {row['question']}\n\nAnswer: {row['answer']}<|endoftext|>"

df['text'] = df.apply(format_example, axis=1)

print("=== Formatted Training Example ===")
print(df.iloc[0]['text'][:400] + "...")

## Step 4: Load Pre-trained Databricks GPT-OSS Model

We'll use **databricks-gpt-oss-120b** - a powerful open-source GPT model from Databricks!

In [ ]:
# Load pre-trained Databricks GPT-OSS model and tokenizer
print(f"Loading {model_name} model...")
print("Note: This is a large model (120B parameters), loading may take a few minutes...")

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16,  # Use FP16 for efficiency
    device_map="auto"  # Automatically distribute across available GPUs
)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f"✓ Model loaded: {model_name}")
print(f"✓ Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")
print(f"✓ Device: {model.device}")

## Step 5: Tokenize the Data

Convert text to tokens that the model can process.

In [ ]:
# Create HuggingFace Dataset
dataset = Dataset.from_pandas(df[['text']])

# Tokenize function
def tokenize_function(examples):
    # Tokenize the text
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        max_length=512,
        padding='max_length',
        return_tensors=None
    )
    # For language modeling, labels are the same as input_ids
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

# Apply tokenization
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=['text'])

print(f"✓ Tokenized {len(tokenized_dataset)} examples")
print(f"✓ Max sequence length: 512 tokens")

## Step 6: Configure Training

Simple supervised fine-tuning with standard cross-entropy loss.

In [ ]:
# Training configuration
output_dir = f"/dbfs/tmp/hedge_fund_qa_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,              # Train for 3 epochs
    per_device_train_batch_size=1,   # Smaller batch for large model
    gradient_accumulation_steps=4,   # Accumulate gradients for effective batch size of 4
    learning_rate=2e-5,              # Lower learning rate for large model
    logging_steps=5,                 # Log every 5 steps
    save_strategy="epoch",           # Save after each epoch
    save_total_limit=2,              # Keep only 2 checkpoints
    report_to="none",                # Disable wandb
    warmup_steps=10,                 # Warmup for stability
    fp16=True,                       # Use mixed precision training
    gradient_checkpointing=True,     # Save memory
)

print("=== Training Configuration ===")
print(f"Model: {model_name}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Mixed precision: {training_args.fp16}")
print(f"Output directory: {output_dir}")

## Step 7: Train the Model!

This uses standard supervised learning with cross-entropy loss.

In [ ]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Start training with MLflow tracking
mlflow.set_experiment(f"/Users/{spark.conf.get('spark.databricks.user.email')}/hedge_fund_qa_finetuning")

print("\n" + "="*60)
print("🚀 Starting Training...")
print(f"Model: {model_name}")
print("="*60 + "\n")

with mlflow.start_run(run_name=f"databricks_gpt_oss_{datetime.now().strftime('%Y%m%d_%H%M')}") as run:
    # Log parameters
    mlflow.log_param("model", model_name)
    mlflow.log_param("num_examples", len(df))
    mlflow.log_param("epochs", training_args.num_train_epochs)
    mlflow.log_param("learning_rate", training_args.learning_rate)
    mlflow.log_param("batch_size", training_args.per_device_train_batch_size)
    mlflow.log_param("gradient_accumulation", training_args.gradient_accumulation_steps)
    
    # Train!
    trainer.train()
    
    print("\n" + "="*60)
    print("✓ Training Complete!")
    print("="*60)
    
    # Save model
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)
    
    mlflow.log_param("output_dir", output_dir)
    print(f"\n✓ Model saved to: {output_dir}")

## Step 8: Test the Fine-tuned Model

Let's see how it performs on new questions!

In [ ]:
# Test questions
test_questions = [
    "What's your view on investing in semiconductor stocks?",
    "How do I calculate portfolio beta?",
    "Should I use stop-loss orders in my trading strategy?",
]

print("="*60)
print("Testing Fine-tuned Model")
print("="*60)

for i, question in enumerate(test_questions, 1):
    print(f"\n📊 Test {i}")
    print(f"Q: {question}")
    print("-" * 60)
    
    # Format input
    prompt = f"Question: {question}\n\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Generate response
    outputs = model.generate(
        inputs.input_ids,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Decode and extract answer
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_response.split("Answer:")[-1].strip()
    
    print(f"A: {answer}\n")

## Step 9: Save Model Metadata to Unity Catalog

In [ ]:
# Create metadata table if it doesn't exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.finetuned_models (
    model_name STRING,
    base_model STRING,
    training_date TIMESTAMP,
    num_examples INT,
    model_path STRING,
    mlflow_run_id STRING,
    use_case STRING,
    notes STRING
)
""")

# Save model metadata
metadata_df = spark.createDataFrame([{
    "model_name": "hedge_fund_qa_gpt2",
    "base_model": model_name,
    "training_date": datetime.now(),
    "num_examples": len(df),
    "model_path": output_dir,
    "mlflow_run_id": run.info.run_id,
    "use_case": "Hedge Fund Q&A",
    "notes": "Simple supervised fine-tuning on 20 hedge fund analyst Q&A pairs"
}])

metadata_df.write.mode("append").saveAsTable(f"{catalog}.{schema}.finetuned_models")

print(f"✓ Model metadata saved to {catalog}.{schema}.finetuned_models")
print(f"✓ MLflow run ID: {run.info.run_id}")

## Step 10: View Model Registry

In [ ]:
# View all registered models
display(spark.table(f"{catalog}.{schema}.finetuned_models"))

## 📊 Summary

### What We Did:
1. ✅ Loaded **20 labeled Q&A pairs** (question = input, answer = label)
2. ✅ Fine-tuned **Databricks GPT-OSS-120B** using **supervised learning**
3. ✅ Used **cross-entropy loss** to train model to predict answers
4. ✅ Tested on new questions
5. ✅ Saved to Unity Catalog at `users.yasamin_tari.finetuned_models`

### The Training Process:
- **Model**: databricks-gpt-oss-120b (120B parameters)
- **Input**: Question text
- **Label**: Expert answer text  
- **Loss**: Cross-entropy between predicted tokens and actual answer tokens
- **Optimization**: Adam optimizer with FP16 mixed precision
- **Memory optimization**: Gradient accumulation + gradient checkpointing

### Next Steps:
- Add more training examples (50-100+)
- Deploy as Model Serving endpoint
- Add validation set for hyperparameter tuning
- Implement LoRA for even more efficient fine-tuning

---

**Model**: `databricks-gpt-oss-120b`  
**Model Location**: `{output_dir}`  
**Registry**: `users.yasamin_tari.finetuned_models`

## 🔍 Understanding What Happened

### The Labeled Dataset

Each training example consists of:
- **Input (X)**: A hedge fund question
- **Label (Y)**: The expert answer we want the model to learn

Example:
```
X: "What's your outlook on tech stocks?"
Y: "Technology sector shows strong momentum... AI infrastructure... recommend 25% allocation..."
```

### The Training Process

1. **Forward Pass**: Model predicts next tokens given the question
2. **Loss Calculation**: Compare predicted tokens vs actual answer tokens (cross-entropy)
3. **Backward Pass**: Calculate gradients 
4. **Weight Update**: Adjust model parameters to reduce loss
5. **Repeat** for all examples, multiple epochs

### What the Model Learned

The base **Databricks GPT-OSS-120B** already knows general language and reasoning. After fine-tuning, it learned:
- Hedge fund terminology (Sharpe ratio, VaR, DCF, etc.)
- Financial metrics and calculations
- Investment recommendation patterns
- Risk analysis frameworks
- Professional analyst communication style

This is **transfer learning**: start with powerful pre-trained model, adapt to specialized hedge fund domain.